# MOMBAK - Jatoba Project

## Import modules

In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm

base_path = r'/Users/estebangaitan/Documents/Github/SMEW'
repo_path = base_path
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "wrappers_postprocessing" / "zeroex_input_data_wrapper.py").is_file()
)
sys.path.insert(0, str(repo_root / "wrappers_postprocessing"))
sys.path.insert(0, str(Path(base_path) / "Projects" / "MOMBAK"))

from porewater_ic import control_porewater, porewater_ic
import smew
from smew import ledger, harness
import zeroex_input_data_wrapper as wp
#import zeroex_input_data_wrapper_v2 as wp2

## MOMBAK - Field data

In [ ]:
# Load field data
# Second data batch 
df_raw_soil_2 = pd.read_csv('/Users/estebangaitan/Documents/Projects/MOMBAK/Jatoba/Soil/JAT_soil_long_BL_BLP_RP1.csv')

df_raw_soil = pd.read_excel('/Users/estebangaitan/Documents/Github/SMEW/Projects/MOMBAK/Field_data/Jatoba_ERW_Soil_Master_Dataset_2026-08-20.xlsx',sheet_name='Master_Data')

# Separate types and periods – Soil
BASELINE = df_raw_soil[df_raw_soil['sampling_event']=='Baseline']
CONTROL = df_raw_soil[df_raw_soil['area_class']=='Control']
TREATMENT = df_raw_soil[(df_raw_soil['area_class']=='Treatment') | (df_raw_soil['area_class']=='Deployment') ]



# Separate types and periods – Porewater
df_raw_water = pd.read_excel('/Users/estebangaitan/Documents/Github/SMEW/Projects/MOMBAK/Field_data/Jatoba_Lysimeter_Water_Chemistry_Consolidated.xlsx', sheet_name='Pivoted',header=1)

df_raw_water['Type'] = np.where((df_raw_water['Lysimeter_ID'].str.contains('MBK3')) | 
                          (df_raw_water['Lysimeter_ID'].str.contains('MBK4')) | 
                          (df_raw_water['Lysimeter_ID'].str.contains('MBK24'))| 
                          (df_raw_water['Lysimeter_ID'].str.contains('MBK25'))|
                          (df_raw_water['Lysimeter_ID'].str.contains('MBK26'))| 
                          (df_raw_water['Lysimeter_ID'].str.contains('MBK27'))| 
                          (df_raw_water['Lysimeter_ID'].str.contains('ZEX3')) |
                          (df_raw_water['Lysimeter_ID'].str.contains('ZEX4')) | 
                          (df_raw_water['Lysimeter_ID'].str.contains('ZEX8')) | 
                          (df_raw_water['Lysimeter_ID'].str.contains('ZEX9')) |
                          (df_raw_water['Lysimeter_ID'].str.contains('ZEX10')),
                          'Control', 'Treatment')


# Deployment date – 1 November 2025
ref = pd.Timestamp('2025-11-01')

# dates in the right format
df_raw_water['Collection_date_ISO'] = pd.to_datetime(df_raw_water['Collection_date_ISO'], format='%Y-%m-%d')
BASELINE['lab_entry_date_agronomic'] = pd.to_datetime(BASELINE['lab_entry_date_agronomic'], format='%Y-%m-%d')


# days since deployment, computed on the full dataframes
df_raw_water['days_since_start'] = (df_raw_water['Collection_date_ISO'] - ref).dt.days
BASELINE['days_since_start'] = (BASELINE['lab_entry_date_agronomic'] - ref).dt.days


# Control and Treatment (.copy() avoids SettingWithCopyWarning on later edits)
CONTROL_WATER = df_raw_water[df_raw_water['Type'] == 'Control'].copy()
TREATMENT_WATER = df_raw_water[df_raw_water['Type'] == 'Treatment'].copy()

## Porewater

### pH and Alkalinity

In [ ]:
#pH and alkalinity
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1)

#-------------- Figure_config ----------------------

fig, (ax1,ax2) = plt.subplots(2,1,figsize=(15, 7))

hue_order = ['Control','Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 2)))
days = sorted(df_raw_water['days_since_start'].unique())

box_width = 3.0                                  # box width in days
offset = {'Control': -2.0, 'Treatment': 2.0}     # shift each box of a pair, in days

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'pH (laboratory)'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax1.set_xlim(0, 365)
ax1.set_xticks(np.arange(0, 366, 30))
ax1.set_xlabel('Days')
ax1.set_ylabel('pH')
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator())
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Total alkalinity'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax2.set_xlim(0, 365)
ax2.set_xticks(np.arange(0, 366, 30))
ax2.set_xlabel('Days')
ax2.set_ylabel('Alkalinity [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')


plt.tight_layout()
plt.savefig('Field_data/Figures/pH_alk_porewater',dpi=150)
plt.show()


### Total Cations

In [ ]:
#Cations
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3,ax4) = plt.subplots(4,1,figsize=(15, 15))

hue_order = ['Control','Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 2)))
days = sorted(df_raw_water['days_since_start'].unique())

box_width = 3.0                                  # box width in days
offset = {'Control': -2.0, 'Treatment': 2.0}     # shift each box of a pair, in days

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved calcium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax1.set_xlim(0, 365)
ax1.set_xticks(np.arange(0, 366, 30))
ax1.set_xlabel('Days')
ax1.set_ylabel('Calcium [mg/L]')
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator())
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved magnesium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax2.set_xlim(0, 365)
ax2.set_xticks(np.arange(0, 366, 30))
ax2.set_xlabel('Days')
ax2.set_ylabel('Magnesium [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# -------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved potassium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax3.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
ax3.set_ylim(0, 50)
ax3.set_xlim(0, 365)
ax3.set_xticks(np.arange(0, 366, 30))
ax3.set_xlabel('Days')
ax3.set_ylabel('Potasium [mg/L]')
ax3.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

#-------------- Panel 4 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved sodium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax4.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
ax4.set_ylim(0, 50)
ax4.set_xlim(0, 365)
ax4.set_xticks(np.arange(0, 366, 30))
ax4.set_xlabel('Days')
ax4.set_ylabel('Sodium [mg/L]')
ax4.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

plt.tight_layout()
plt.savefig('Field_data/Figures/cations_porewater',dpi=150)
plt.show()


### Anions

In [ ]:
#Anions
sns.set_theme("notebook", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1.2)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3) = plt.subplots(3,1,figsize=(15, 12))

hue_order = ['Control','Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 2)))
days = sorted(df_raw_water['days_since_start'].unique())

box_width = 3.0                                  # box width in days
offset = {'Control': -2.0, 'Treatment': 2.0}     # shift each box of a pair, in days

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Chloride'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax1.set_xlim(0, 365)
ax1.set_xticks(np.arange(0, 366, 30))
ax1.set_xlabel('Days')
ax1.set_ylabel('Chloride [mg/L]')
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator())
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Nitrate (as N)'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

ax2.set_xlim(0, 365)
ax2.set_xticks(np.arange(0, 366, 30))
ax2.set_xlabel('Days')
ax2.set_ylabel('Nitrate (as N) [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# -------------- Panel 3 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Sulfate'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax3.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
#ax3.set_ylim(0, 50)
ax3.set_xlim(0, 365)
ax3.set_xticks(np.arange(0, 366, 30))
ax3.set_xlabel('Days')
ax3.set_ylabel('Sulfate [mg/L]')
ax3.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

plt.tight_layout()
plt.savefig('Field_data/Figures/anions_porewater',dpi=150)
plt.show()


## Soil Data

In [ ]:
#Baseline OC, pH, 
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1.2)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3) = plt.subplots(1,3,figsize=(12, 3))

# –––––––– Panel 1 ––––––––––––
sns.boxplot(
    data=BASELINE, x='area_class', y='ph_cacl2',
    hue='area_class', hue_order=['Control','Deployment','Treatment'], palette='pastel',
    order=['Control','Deployment','Treatment'], width=0.2,
    linewidth=1, ax=ax1
)

ax1.set_xlabel('')
ax1.set_ylabel(f'pH')
ax1.set_ylim(3,9)

# –––––––– Panel 2 ––––––––––––
sns.boxplot(data=BASELINE,y='organic_c_combustion_pct',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax2)
ax2.set_xlabel('')
ax2.set_ylabel(f'Organic Carbon %')
ax2.set_ylim(0,10)

# –––––––– Panel 3 ––––––––––––
means = (BASELINE.groupby("area_class")[["clay_pct", "silt_pct", "sand_pct"]]
         .mean().loc[["Control", "Treatment", "Deployment"]])
bottom = 0
for col in means.columns:
    ax3.bar(means.index, means[col], bottom=bottom, alpha=0.8, label=col)
    bottom = bottom + means[col]

ax3.legend(title="Fraction", frameon=True)
ax3.set_ylabel('Soil Texture %')
plt.tight_layout()
plt.savefig('Field_data/Figures/ph_OC_texture.jpeg',dpi=150)

In [ ]:
#Baseline CEC
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3,ax4,ax5) = plt.subplots(1,5,figsize=(15, 3))

# –––––––– Panel 1 ––––––––––––
sns.boxplot(data=BASELINE,y='base_saturation_pct',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax1)
ax1.set_xlabel('')
ax1.set_ylabel(f'Base Saturation %')
ax1.set_ylim(0,100)

# –––––––– Panel 2 ––––––––––––
sns.boxplot(data=BASELINE,y='total_cec_ph7_cmolc_dm3',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax2)
ax2.set_xlabel('')
ax2.set_ylabel(f'Total CEC [$cmolc/dm^3$]')
ax2.set_ylim(0,15)

# –––––––– Panel 3 ––––––––––––
sns.boxplot(data=BASELINE,y='exch_ca_cmolc_dm3',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax3)
ax3.set_xlabel('')
ax3.set_ylabel(f'Ca [$cmolc/dm^3$]')
ax3.set_ylim(0,15)

# –––––––– Panel 4 ––––––––––––
sns.boxplot(data=BASELINE,y='exch_mg_cmolc_dm3',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax4)
ax4.set_xlabel('')
ax4.set_ylabel(f'Mg [$cmolc/dm^3$]')
ax4.set_ylim(0,15)

# –––––––– Panel 5 ––––––––––––
sns.boxplot(data=BASELINE,y='potential_acidity_H_Al_cmolc_dm3',x='area_class',hue='area_class',order=['Control','Deployment','Treatment'],width=0.2,ax=ax5)
ax5.set_xlabel('')
ax5.set_ylabel(f'Al + H [$cmolc/dm^3$]')
ax5.set_ylim(0,15)

plt.suptitle('Exchangeable Fraction')
plt.tight_layout()
plt.savefig('Field_data/Figures/Exchangeable_Fraction.jpeg',dpi=150)

In [ ]:
PLOT_DATA = df_raw_soil.loc[
    df_raw_soil['sampling_event'].isin(['Baseline', 'RP1'])
].copy()

PLOT_DATA['lab_entry_date_agronomic'] = pd.to_datetime(
    PLOT_DATA['lab_entry_date_agronomic']
)
PLOT_DATA['days_since_start'] = (
    PLOT_DATA['lab_entry_date_agronomic'] - ref
).dt.days

days = sorted(PLOT_DATA['days_since_start'].dropna().unique())

# Include earlier PLOT_DATA days and leave room for the boxes around each day.
x_min = min(days) - 6
x_max = max(days) + 6
x_ticks = np.arange(
    int(np.floor(x_min / 30) * 30),
    int(np.ceil(365 / 30) * 30) + 1,
    180
)

hue_order = ['Control', 'Treatment','Deployment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 3)))
days = sorted(PLOT_DATA['days_since_start'].unique())
box_width = 4.0                                  # box width in days
offset = {'Control': -6.0, 'Treatment':0, 'Deployment': 6.0}     # shift each box of a pair, in days


def boxplot_panel(ax, column, ylabel, top_dates=False):
    for h in hue_order:
        sub = PLOT_DATA[PLOT_DATA['area_class'] == h]
        data, pos = [], []
        for d in days:
            vals = sub.loc[sub['days_since_start'] == d, column].dropna()
            if len(vals):
                data.append(vals.values)
                pos.append(d + offset[h])
        bp = ax.boxplot(
            data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
            flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
            medianprops=dict(color='black'),
        )
        for box in bp['boxes']:
            box.set_facecolor(colors[h])
            box.set_alpha(0.5)

    ax.set_xlim(x_min, 365)
    ax.set_xticks(x_ticks)
    ax.set_xlabel('Days')
    ax.set_ylabel(ylabel)
    ax.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Plot Type')

    if top_dates:
        # Top axis: calendar dates, day 0 = 2025-10-01
        base = mdates.date2num(ref)
        top = ax.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
        top.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        top.set_xlabel('Date')


#-------------- Figure 1: pH, Total CEC, Base saturation ----------------------
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8,8))

boxplot_panel(ax1, 'ph_cacl2', r'pH (CaCl$_2$)', top_dates=True)
boxplot_panel(ax2, 'total_cec_ph7_cmolc_dm3', r'Total CEC [$cmolc/dm^3$]')
boxplot_panel(ax3, 'base_saturation_pct', r'Base Saturation %')

plt.tight_layout()
plt.savefig('Field_data/Figures/pH_CEC_BaseSat_evolution.jpeg', dpi=150)
plt.show()

#-------------- Figure 2: Ca, Mg, Al + H ----------------------
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8,8))

boxplot_panel(ax1, 'exch_ca_cmolc_dm3', r'Ca [$cmolc/dm^3$]', top_dates=True)
boxplot_panel(ax2, 'exch_mg_cmolc_dm3', r'Mg [$cmolc/dm^3$]')
boxplot_panel(ax3, 'potential_acidity_H_Al_cmolc_dm3', r'Al + H [$cmolc/dm^3$]')

plt.tight_layout()
plt.savefig('Field_data/Figures/Exchangeable_Cations_evolution.jpeg', dpi=150)
plt.show()

## MOMBAK - SMEW

In [ ]:
# Generate rainfall data – lambda and alpha values for the stochastic rainfall model.
smew.stoch_lambda_alpha(csv_path="/Users/estebangaitan/Documents/Github/SMEW/Projects/MOMBAK/Field_data/CEMADEN_Estiva_Gerbi_Centro_2024_monthly.csv")
conc_obs, pH_obs = control_porewater()  

### Control


In [ ]:
inp_ctl = wp.read_input_data("MOMBAK", "Control")
with porewater_ic(conc_obs):
    out_ctl = wp.run_SMEW("MOMBAK", inp_ctl, seed=1, interpolate_frozen=False)
    
ctl_carb_alk = (out_ctl['HCO3'] + 2*out_ctl['CO3'])*61.02e-3 # titration alkalinity
ca_ctl = out_ctl['Ca'] * 40.08e-3 # to mg/L
na_ctl = out_ctl['Na'] * 22.99e-3 # to mg/L
mg_ctl = out_ctl['Mg'] * 24.31e-3 # to mg/L
k_ctl = out_ctl['K'] * 39.10e-3 # to mg/L

### Treatment

In [ ]:
inp_trt = wp.read_input_data("MOMBAK", "Jatoba_Farm")
with porewater_ic(conc_obs):
    
    out_trt = wp.run_SMEW("MOMBAK", inp_trt, seed=1, interpolate_frozen=False)

trt_carb_alk = (out_trt['HCO3'] + 2*out_trt['CO3'])*61.02e-3 # titration alkalinity
ca_trt = out_trt['Ca'] * 40.08e-3 # to mg/L
na_trt = out_trt['Na'] * 22.99e-3 # to mg/L
mg_trt = out_trt['Mg'] * 24.31e-3 # to mg/L
k_trt = out_trt['K'] * 39.10e-3 # to mg/L

## Simulation V1

In [ ]:
project = 'MOMBAK'
field_name = 'Jatoba_Farm'

input_data = wp.read_input_data(project_name=project,value_col=field_name)
output_data = wp.run_SMEW(project_name=project,input_data=input_data)
wp.create_plots(project_name=project,model_results=output_data,out_dir='runs/'+field_name)  # runs/ is gitignored
led = ledger.build(output_data)        # water= and oc= are optional
print(ledger.report(led))              # per-pool sources/sinks, closure per step and over the run
print(ledger.passed(led))              # True / False

# evolution of the run: save a thinned snapshot
harness.verify_contract(output_data)
payload = harness.collect(output_data, output_data['t'], n_out=500)
harness.save("runs/mombak_run1.npz", payload)

# conversion outputs
carb_alk = output_data['HCO3'] + 2*output_data['CO3']*61.02e-3 # titration alkalinity
ca_smew = output_data['Ca'] * 40.08e-3 # to mg/L
na_smew = output_data['Na'] * 22.99e-3 # to mg/L
mg_smew = output_data['Mg'] * 24.31e-3 # to mg/L
k_smew = output_data['K'] * 39.10e-3 # to mg/L

## SMEW batch SSA

In [ ]:
batch = wp2.run_batch('MOMBAK', 'Jatoba_Farm',
                      scenarios={'SSA_in':[0.1,1,10]}, 
                      seed=1, plot=True)

batch.summary              # one row per run

## SMEW vs Porewater

In [ ]:
PLOT_DATA = df_raw_soil.loc[
    df_raw_soil['sampling_event'].isin(['Baseline', 'RP1'])
].copy()

PLOT_DATA['lab_entry_date_agronomic'] = pd.to_datetime(
    PLOT_DATA['lab_entry_date_agronomic']
)
PLOT_DATA['days_since_start'] = (
    PLOT_DATA['lab_entry_date_agronomic'] - ref
).dt.days

days = sorted(PLOT_DATA['days_since_start'].dropna().unique())

# Include earlier PLOT_DATA days and leave room for the boxes around each day.
x_min = min(days) - 6
x_max = max(days) + 6
x_ticks = np.arange(
    int(np.floor(x_min / 30) * 30),
    int(np.ceil(1095 / 30) * 30) + 1,
    30
)

hue_order = ['Control', 'Deployment','Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 3)))
days = sorted(PLOT_DATA['days_since_start'].unique())
box_width = 10                                  # box width in days
offset = {'Control': -10.0, 'Deployment':0, 'Treatment': 10.0}     # shift each box of a pair, in days

fig, (ax1,ax2) = plt.subplots(2,1,figsize=(10, 7))

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = PLOT_DATA[PLOT_DATA['area_class'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'ph_cacl2'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)

smew_ph_ctl, = ax1.plot(out_ctl['t'],out_ctl['pH'],c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_ph_trt, = ax1.plot(out_trt['t'],out_trt['pH'],c='salmon',linestyle='--',label='SMEW - Treatment')

ax1.set_xlim(x_min, 1095)
ax1.set_xticks(np.arange(0, 1095, 120))
ax1.set_xlabel('Days')
ax1.set_ylabel(r"pH")
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Plot Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------
hue_pore = ['Control', 'Treatment']
colors_pore = dict(zip(hue_pore, sns.color_palette('pastel', 2)))
days_pore = sorted(df_raw_water['days_since_start'].unique())
box_width_pore = 10                                  # box width in days
offset_pore = {'Control': -6.0,  'Treatment': 6}

for h in hue_pore:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days_pore:
        vals = sub.loc[sub['days_since_start'] == d, 'Total alkalinity'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset_pore[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width_pore, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors_pore[h], markeredgecolor=colors_pore[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors_pore[h])
        box.set_alpha(0.5)
smew_alk_ctl, = ax2.plot(out_ctl['t'],ctl_carb_alk,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_alk_trt, = ax2.plot(out_trt['t'],trt_carb_alk,c='salmon',linestyle='--',label='SMEW - Treatment')


ax2.set_xlim(x_min, 1095)
ax2.set_xticks(np.arange(0, 1095, 120))
ax2.set_xlabel('Days')
ax2.set_ylabel(r'Alkalinity [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors_pore[h], alpha=0.5, label=h) for h in hue_pore], title='Plot Type')


plt.tight_layout()
#plt.savefig('Field_data/Figures/Exchangeable_Fraction_evolution.jpeg',dpi=150)
plt.show()

In [ ]:
#Cations
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3,ax4) = plt.subplots(4,1,figsize=(15, 15))

hue_order = ['Control', 'Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 2)))
days = sorted(df_raw_water['days_since_start'].unique())

box_width = 10                                  # box width in days
offset = {'Control': -6.0, 'Treatment': 6.0}     # shift each box of a pair, in days

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved calcium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_ca_ctl, = ax1.plot(out_ctl['t'],ca_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_ca_trt, = ax1.plot(out_trt['t'],ca_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax1.set_xlim(0, 1095)
ax1.set_xticks(np.arange(0, 1095, 120))
ax1.set_xlabel('Days')
ax1.set_ylabel('Calcium [mg/L]')
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved magnesium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_mg_ctl, = ax2.plot(out_ctl['t'],mg_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_mg_trt, = ax2.plot(out_trt['t'],mg_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax2.set_xlim(0, 1095)
ax2.set_xticks(np.arange(0, 1095, 120))
ax2.set_xlabel('Days')
ax2.set_ylabel('Magnesium [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# -------------- Panel 3 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved potassium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax3.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_k_ctl, = ax3.plot(out_ctl['t'],k_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_k_trt, = ax3.plot(out_trt['t'],k_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax3.set_xlim(0, 1095)
ax3.set_xticks(np.arange(0, 1095, 120))
ax3.set_xlabel('Days')
ax3.set_ylabel('Potasium [mg/L]')
ax3.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

#-------------- Panel 4 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved sodium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax4.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_na_ctl, = ax4.plot(out_ctl['t'],na_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_na_trt, = ax4.plot(out_trt['t'],na_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax4.set_xlim(0, 1095)
ax4.set_xticks(np.arange(0, 1095, 120))
ax4.set_xlabel('Days')
ax4.set_ylabel('Sodium [mg/L]')
ax4.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

plt.tight_layout()
plt.show()

### Anions

In [ ]:
#Anions
sns.set_theme("paper", style="whitegrid", palette="pastel", font='Helvetica', font_scale=1)

#-------------- Figure_config ----------------------

fig, (ax1,ax2,ax3) = plt.subplots(3,1,figsize=(15, 12))

hue_order = ['Control', 'Treatment']
colors = dict(zip(hue_order, sns.color_palette('pastel', 2)))
days = sorted(df_raw_water['days_since_start'].unique())

box_width = 10                                  # box width in days
offset = {'Control': -6.0, 'Treatment': 6.0}     # shift each box of a pair, in days

#-------------- Panel 1 ----------------------
for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved calcium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax1.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_ca_ctl, = ax1.plot(out_ctl['t'],ca_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_ca_trt, = ax1.plot(out_trt['t'],ca_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax1.set_xlim(0, 1095)
ax1.set_xticks(np.arange(0, 366, 30))
ax1.set_xlabel('Days')
ax1.set_ylabel('Calcium [mg/L]')
ax1.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# Top ax1is: calendar dates, day 0 = 2025-10-01
base = mdates.date2num(ref)
top = ax1.secondary_xaxis('top', functions=(lambda d: base + d, lambda m: m - base))
top.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
top.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
top.set_xlabel('Date')

#-------------- Panel 2 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved magnesium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax2.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_mg_ctl, = ax2.plot(out_ctl['t'],mg_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_mg_trt, = ax2.plot(out_trt['t'],mg_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax2.set_xlim(0, 1095)
ax2.set_xticks(np.arange(0, 366, 30))
ax2.set_xlabel('Days')
ax2.set_ylabel('Magnesium [mg/L]')
ax2.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

# -------------- Panel 3 ----------------------

for h in hue_order:
    sub = df_raw_water[df_raw_water['Type'] == h]
    data, pos = [], []
    for d in days:
        vals = sub.loc[sub['days_since_start'] == d, 'Dissolved potassium'].dropna()
        if len(vals):
            data.append(vals.values)
            pos.append(d + offset[h])
    bp = ax3.boxplot(
        data, positions=pos, widths=box_width, patch_artist=True, manage_ticks=False,
        flierprops=dict(marker='o', markersize=6, markerfacecolor=colors[h], markeredgecolor=colors[h]),
        medianprops=dict(color='black'),
    )
    for box in bp['boxes']:
        box.set_facecolor(colors[h])
        box.set_alpha(0.5)
smew_k_ctl, = ax3.plot(out_ctl['t'],k_ctl,c='cornflowerblue',linestyle='--',label='SMEW - Control')
smew_k_trt, = ax3.plot(out_trt['t'],k_trt,c='salmon',linestyle='--',label='SMEW - Treatment')

ax3.set_xlim(0, 1095)
ax3.set_xticks(np.arange(0, 366, 30))
ax3.set_xlabel('Days')
ax3.set_ylabel('Potasium [mg/L]')
ax3.legend(handles=[Patch(facecolor=colors[h], alpha=0.5, label=h) for h in hue_order], title='Type')

plt.tight_layout()
plt.show()

# Test GeoPandas

In [ ]:
XLSX = "/Users/estebangaitan/Documents/Projects/MOMBAK/Jatoba/Soil/Jatoba_ERW_Soil_Master_Dataset_2026-08-20.xlsx"
OUT = sys.argv[1] if len(sys.argv) > 1 else "bs_hal_maps.png"
DEPTH = "0-15 cm"          # or "15-30 cm"
CRS_UTM = "EPSG:31983"     # SIRGAS 2000 / UTM 23S -> metres
CELL, POWER, K = 10, 2, 12  # grid cell (m), IDW power, nearest neighbours
EVENTS = ["Baseline", "Post-Baseline", "RP1"]
VARS = {
    "base_saturation_pct": dict(label="Base saturation V (%)", cmap="YlGnBu", dcmap="RdBu"),
    "potential_acidity_H_Al_cmolc_dm3": dict(label="H+Al (cmolc/dm³)", cmap="YlOrRd", dcmap="RdBu_r"),
}

m = pd.read_excel(XLSX, sheet_name="Master_Data")
m = m[(m.sample_class == "Primary") & (m.depth_interval == DEPTH)]
pts = gpd.GeoDataFrame(
    m, geometry=gpd.points_from_xy(m.longitude_wgs84, m.latitude_wgs84), crs="EPSG:4326"
).to_crs(CRS_UTM)
pts["x"], pts["y"] = pts.geometry.x, pts.geometry.y

# one grid for all panels, clipped to the sampled area (buffered convex hull)
reg = pts.drop_duplicates("point_id")
hull = reg.geometry.union_all().convex_hull.buffer(40)
x0, y0, x1, y1 = hull.bounds
gx, gy = np.meshgrid(np.arange(x0, x1, CELL), np.arange(y0, y1, CELL))
grid = gpd.GeoSeries(gpd.points_from_xy(gx.ravel(), gy.ravel()), crs=CRS_UTM)
inside = grid.within(hull).to_numpy().reshape(gx.shape)


def idw(xy, z):
    d = np.hypot(gx.ravel()[:, None] - xy[:, 0], gy.ravel()[:, None] - xy[:, 1])
    k = min(K, len(z))
    idx = np.argpartition(d, k - 1, axis=1)[:, :k]
    dk = np.take_along_axis(d, idx, 1)
    w = 1.0 / np.maximum(dk, 1.0) ** POWER
    out = ((w * z[idx]).sum(1) / w.sum(1)).reshape(gx.shape)
    return np.where(inside, out, np.nan)


def surface(col, event):
    s = pts[pts.sampling_event == event].dropna(subset=[col])
    if s.empty:
        return None
    return idw(s[["x", "y"]].to_numpy(), s[col].to_numpy(float)), s


ext = (x0, x1, y0, y1)
fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)
for r, (col, cfg) in enumerate(VARS.items()):
    surf = {e: surface(col, e) for e in EVENTS}
    vals = np.concatenate([s[1][col].to_numpy() for s in surf.values() if s])
    vmin, vmax = np.nanpercentile(vals, [1, 99])  # shared scale across periods
    for c, e in enumerate(EVENTS):
        ax = axes[r, c]
        if surf[e] is None:
            ax.text(0.5, 0.5, "no data\n(4-acid digestion only)", ha="center", va="center",
                    transform=ax.transAxes, color="0.4")
        else:
            z, s = surf[e]
            im = ax.imshow(z, origin="lower", extent=ext, cmap=cfg["cmap"], vmin=vmin,vmax=vmax)
            ax.scatter(s.x, s.y, s=6, c="k", alpha=0.5, lw=0)
        ax.set_title(e); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes[r, :3], shrink=0.8, label=cfg["label"])

    # change map: RP1 - Baseline at points present in both
    p = pts.pivot_table(index="point_id", columns="sampling_event", values=col)
    d = (p["RP1"] - p["Baseline"]).dropna()
    xy = reg.set_index("point_id").loc[d.index, ["x", "y"]].to_numpy()
    lim = np.nanpercentile(np.abs(d), 95)
    ax = axes[r, 3]
    im = ax.imshow(idw(xy, d.to_numpy()), origin="lower", extent=ext, cmap=cfg["dcmap"],
                   norm=TwoSlopeNorm(0, -lim, lim))
    ax.scatter(xy[:, 0], xy[:, 1], s=6, c="k", alpha=0.5, lw=0)
    ax.set_title("RP1 − Baseline"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, shrink=0.8, label="Δ " + cfg["label"])

fig.suptitle(f"Jatoba — {DEPTH}, IDW interpolation of 120 points (k={K}, p={POWER}")

In [ ]:
XLSX = "/Users/estebangaitan/Documents/Projects/MOMBAK/Jatoba/Soil/Jatoba_ERW_Soil_Master_Dataset_2026-08-20.xlsx"
OUT = sys.argv[1] if len(sys.argv) > 1 else "bs_hal_maps_groups.png"
DEPTH = "0-15 cm"          # or "15-30 cm"
CRS_UTM = "EPSG:31983"     # SIRGAS 2000 / UTM 23S (metres)
CELL, POWER, K, HULL_BUF = 10, 2, 12, 40
EVENTS = ["Baseline", "RP1"]
GROUPS = {"Control": ["Control"], "Treatment + Deployment": ["Treatment", "Deployment"]}
VARS = {
    "base_saturation_pct": dict(label="Base saturation V (%)", cmap="YlGnBu", dcmap="RdBu"),
    "potential_acidity_H_Al_cmolc_dm3": dict(label="H+Al (cmolc/dm³)", cmap="YlOrRd", dcmap="RdBu_r"),
}

m = pd.read_excel(XLSX, sheet_name="Master_Data")
m = m[(m.sample_class == "Primary") & (m.depth_interval == DEPTH) & m.sampling_event.isin(EVENTS)]
pts = gpd.GeoDataFrame(
    m, geometry=gpd.points_from_xy(m.longitude_wgs84, m.latitude_wgs84), crs="EPSG:4326"
).to_crs(CRS_UTM)
pts["x"], pts["y"] = pts.geometry.x, pts.geometry.y
reg = pts.drop_duplicates("point_id").set_index("point_id")

x0, y0, x1, y1 = reg.geometry.union_all().convex_hull.buffer(HULL_BUF).bounds
gx, gy = np.meshgrid(np.arange(x0, x1, CELL), np.arange(y0, y1, CELL))
grid = gpd.GeoSeries(gpd.points_from_xy(gx.ravel(), gy.ravel()), crs=CRS_UTM)
ext = (x0, x1, y0, y1)


def idw(xy, z):
    """IDW surface from group points, clipped to that group's own buffered hull."""
    hull = gpd.GeoSeries(gpd.points_from_xy(xy[:, 0], xy[:, 1])).union_all().convex_hull.buffer(HULL_BUF)
    inside = grid.within(hull).to_numpy().reshape(gx.shape)
    d = np.hypot(gx.ravel()[:, None] - xy[:, 0], gy.ravel()[:, None] - xy[:, 1])
    k = min(K, len(z))
    idx = np.argpartition(d, k - 1, axis=1)[:, :k]
    w = 1.0 / np.maximum(np.take_along_axis(d, idx, 1), 1.0) ** POWER
    out = ((w * z[idx]).sum(1) / w.sum(1)).reshape(gx.shape)
    return np.where(inside, out, np.nan)


def style(ax, title):
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.set_xticks([]); ax


fig, axes = plt.subplots(4, 3, figsize=(13, 17), constrained_layout=True)
row = 0
for col, cfg in VARS.items():
    # shared colour scale across both groups and both events
    v = pts[col].dropna()
    vmin, vmax = np.percentile(v, [1, 99])
    piv = pts.pivot_table(index="point_id", columns="sampling_event", values=col)
    dall = (piv["RP1"] - piv["Baseline"]).dropna()
    lim = np.percentile(np.abs(dall), 95)
    for gname, classes in GROUPS.items():
        ids = reg.index[reg.area_class.isin(classes)]
        for c, e in enumerate(EVENTS):
            s = pts[(pts.sampling_event == e) & pts.point_id.isin(ids)].dropna(subset=[col])
            ax = axes[row, c]
            im = ax.imshow(idw(s[["x", "y"]].to_numpy(), s[col].to_numpy(float)), origin="lower",
                           extent=ext, cmap=cfg["cmap"], vmin=vmin, vmax=vmax)
            ax.scatter(s.x, s.y, s=7, c="k", alpha=0.55, lw=0)
            style(ax, f"{gname}\n{e} (n={len(s)}, mean {s[col].mean():.2f})")
        if row % 2 == 1:
            fig.colorbar(im, ax=axes[row - 1:row + 1, :2], shrink=0.6, label=cfg["label"])
        d = dall.loc[dall.index.intersection(ids)]
        xy = reg.loc[d.index, ["x", "y"]].to_numpy()
        ax = axes[row, 2]
        imd = ax.imshow(idw(xy, d.to_numpy()), origin="lower", extent=ext, cmap=cfg["dcmap"],
                        norm=TwoSlopeNorm(0, -lim, lim))
        ax.scatter(xy[:, 0], xy[:, 1], s=7, c="k", alpha=0.55, lw=0)
        style(ax, f"{gname}\nRP1 − Baseline (mean Δ {d.mean():+.2f})")
        row += 1
    fig.colorbar(imd, ax=axes[row - 2:row, 2], shrink=0.6, label="Δ " + cfg["label"])

fig.suptitle(f"Jatoba — {DEPTH}, IDW (k={K}, p={POWER})", fontsize=13)
